# Safehouse Value-Add Pipeline: Lighthouse Sanctuary

## 1. Problem Framing

### Business problem
Executive and Operations leadership need to know which safehouses **systematically outperform or underperform** after accounting for **case mix** (baseline difficulty of residents), and whether **Operations** funding aligns with that case-adjusted performance.

### Explanatory (value-added) objective — not prediction
This notebook is an **explanatory / causal-inference style** analysis (multiple linear regression for interpretation), consistent with **Chapter 9**: we estimate **conditional associations** while holding other measured factors constant. We are **not** building a model to forecast future health scores for new residents (that predictive framing is **Chapter 11** and is not the goal here).

### Why raw averages mislead
Ranking safehouses by **raw average** outcome change mixes two things: (1) the facility’s contribution and (2) **who** the facility serves. A safehouse that accepts higher-risk residents can look “worse” even if it delivers strong support. Value-added style modeling adds **case-mix controls** and **safehouse indicators** so leadership can separate composition from facility-specific patterns — while remaining honest about **omitted-variable bias** and the limits of observational data.


In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

candidate_dirs = [Path("../../data/raw"), Path("data/raw")]
DATA_DIR = next((p for p in candidate_dirs if p.exists()), Path("../../data/raw"))
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

print(f"Data dir: {DATA_DIR.resolve()} (exists={DATA_DIR.exists()})")


Data dir: C:\Users\User\OneDrive\BYU\WINTER 2026\INTEX\intex2\data\raw (exists=True)


## 2. Data Acquisition, Preparation & Exploration

We load resident, health, safehouse, and allocation tables, then build the **resident–month spine** used in Pipeline 1. The sustained outcome is **`health_score_delta_6m`**: general health at month **T+6** minus general health at month **T**.

We also compute a numeric **`age_upon_admission_years`** from `date_of_birth` and `date_of_admission` (rather than using the human-readable string column in `residents.csv`). **Operations** funding is the sum of `amount_allocated` in `donation_allocations.csv` where `program_area == 'Operations'`.


In [2]:
# --- Load ---
residents = pd.read_csv(DATA_DIR / "residents.csv")
health = pd.read_csv(DATA_DIR / "health_wellbeing_records.csv")
safehouses = pd.read_csv(DATA_DIR / "safehouses.csv")
donation_allocations = pd.read_csv(DATA_DIR / "donation_allocations.csv")

# Parse dates
residents["date_of_birth"] = pd.to_datetime(residents["date_of_birth"], errors="coerce")
residents["date_of_admission"] = pd.to_datetime(residents["date_of_admission"], errors="coerce")
health["record_date"] = pd.to_datetime(health["record_date"], errors="coerce")
health["month"] = health["record_date"].dt.to_period("M").dt.to_timestamp()

# Numeric age at admission (years)
residents["age_upon_admission_years"] = (
    (residents["date_of_admission"] - residents["date_of_birth"]).dt.days / 365.25
)

# Operations budget per safehouse
ops_alloc = donation_allocations.loc[donation_allocations["program_area"].eq("Operations")].copy()
ops_budget_by_sh = (
    ops_alloc.groupby("safehouse_id", as_index=False)["amount_allocated"].sum().rename(
        columns={"amount_allocated": "operations_budget_total"}
    )
)

print("Shapes:", {"residents": residents.shape, "health": health.shape, "safehouses": safehouses.shape, "donation_allocations": donation_allocations.shape})
print("\nOperations budget by safehouse (head):")
print(ops_budget_by_sh.sort_values("safehouse_id").head().to_string())


Shapes: {'residents': (60, 50), 'health': (534, 15), 'safehouses': (9, 13), 'donation_allocations': (521, 7)}

Operations budget by safehouse (head):
   safehouse_id  operations_budget_total
0             1                 10554.82
1             2                  9192.07
2             3                  8398.80
3             4                  5810.55
4             5                  4942.18


In [3]:
# --- Resident–month spine + 6-month delta (same pattern as Pipeline 1) ---
resident_month_ranges = (
    health.groupby("resident_id")["month"].agg(month_min="min", month_max="max").dropna().reset_index()
)

spine_rows = []
for _, row in resident_month_ranges.iterrows():
    for m in pd.date_range(start=row["month_min"], end=row["month_max"], freq="MS"):
        spine_rows.append((row["resident_id"], m))

spine = pd.DataFrame(spine_rows, columns=["resident_id", "month"])

health_monthly = health[["resident_id", "month", "general_health_score"]].copy()
health_t = health_monthly.rename(columns={"general_health_score": "general_health_score_t"})
health_tp6 = health_monthly.copy()
health_tp6["month"] = health_tp6["month"] - pd.DateOffset(months=6)
health_tp6 = health_tp6.rename(columns={"general_health_score": "general_health_score_t_plus_6"})

spine = spine.merge(health_t, on=["resident_id", "month"], how="left")
spine = spine.merge(health_tp6, on=["resident_id", "month"], how="left")
spine["health_score_delta_6m"] = (
    spine["general_health_score_t_plus_6"] - spine["general_health_score_t"]
)

case_mix_cols = [
    "resident_id",
    "safehouse_id",
    "initial_risk_level",
    "case_category",
    "is_pwd",
    "family_is_4ps",
    "age_upon_admission_years",
]
model_df = spine.merge(residents[case_mix_cols], on="resident_id", how="left")

print("Spine rows:", len(spine))
print("Rows with non-null 6-month delta:", model_df["health_score_delta_6m"].notna().sum())
model_df.head()


Spine rows: 534
Rows with non-null 6-month delta: 174


,resident_id,month,general_health_score_t,general_health_score_t_plus_6,health_score_delta_6m,safehouse_id,initial_risk_level,case_category,is_pwd,family_is_4ps,age_upon_admission_years
0,1,2023-10-01,3.09,NaN,NaN,4,Critical,Neglected,False,False,15.126626
1,1,2023-11-01,3.05,NaN,NaN,4,Critical,Neglected,False,False,15.126626
2,1,2023-12-01,3.05,NaN,NaN,4,Critical,Neglected,False,False,15.126626
3,1,2024-01-01,3.08,NaN,NaN,4,Critical,Neglected,False,False,15.126626
4,1,2024-02-01,3.13,NaN,NaN,4,Critical,Neglected,False,False,15.126626


In [4]:
# --- Exploration: naive average delta by safehouse (misleading if case mix differs) ---
eda = model_df.dropna(subset=["health_score_delta_6m", "safehouse_id"]).copy()
naive_by_sh = (
    eda.groupby("safehouse_id", as_index=False)["health_score_delta_6m"]
    .mean()
    .sort_values("health_score_delta_6m")
)
naive_by_sh = naive_by_sh.merge(safehouses[["safehouse_id", "name", "safehouse_code"]], on="safehouse_id", how="left")

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=naive_by_sh, x="safehouse_id", y="health_score_delta_6m", color="steelblue", ax=ax)
ax.set_title("Naive comparison: mean 6-month health score delta by safehouse")
ax.set_xlabel("safehouse_id")
ax.set_ylabel("Mean health_score_delta_6m")
plt.tight_layout()
plt.show()

# Outcome distribution (Chapter 10: skew informs residual behavior)
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(eda["health_score_delta_6m"], bins=20, kde=True, color="teal", ax=ax)
ax.set_title("Distribution of health_score_delta_6m (non-null rows)")
ax.set_xlabel("health_score_delta_6m")
plt.tight_layout()
plt.show()


## 3. Modeling & Feature Selection

We estimate a **multiple linear regression (OLS)** model (**Chapter 9**): the outcome is `health_score_delta_6m`. Predictors include **case-mix controls** and **safehouse fixed effects** implemented with **dummy variables** (`drop_first=True`), so each coefficient compares that safehouse to an **omitted reference** safehouse (the dropped `safehouse_id` dummy, here **safehouse 1**) after conditioning on measured case mix. Categorical case-mix dummies omit **Critical** (risk) and **Abandoned** (case category) as reference levels.

**Feature selection** here is **domain-driven** (per instructions): `initial_risk_level`, `case_category`, `is_pwd`, `family_is_4ps`, and `age_upon_admission_years`, plus `safehouse_id` dummies.

After fitting, we report **Chapter 10** diagnostics focused on explanatory modeling: **VIF** on the **case-mix** design (excluding many safehouse indicators, which can destabilize VIF), **residuals vs fitted**, and a **Q–Q** plot for residual normality as a *signal*, not a mechanical pass/fail.


In [5]:
reg = model_df.dropna(subset=["health_score_delta_6m"]).copy()

# Encode features for statsmodels OLS
X_cat = pd.get_dummies(reg[["initial_risk_level", "case_category"]], drop_first=True)
X_num = pd.DataFrame(
    {
        "is_pwd": reg["is_pwd"].astype(int),
        "family_is_4ps": reg["family_is_4ps"].astype(int),
        "age_upon_admission_years": reg["age_upon_admission_years"].astype(float),
    }
)
X_sh = pd.get_dummies(reg["safehouse_id"], prefix="safehouse_id", drop_first=True)

X = pd.concat([X_num, X_cat, X_sh], axis=1)
X = sm.add_constant(X).astype(float)
y = reg["health_score_delta_6m"].astype(float)

ols_model = sm.OLS(y, X).fit()
print(ols_model.summary())


                              OLS Regression Results                             
Dep. Variable:     health_score_delta_6m   R-squared:                       0.355
Model:                               OLS   Adj. R-squared:                  0.285
Method:                    Least Squares   F-statistic:                     5.058
Date:                   Wed, 08 Apr 2026   Prob (F-statistic):           1.13e-08
Time:                           21:21:25   Log-Likelihood:                 57.805
No. Observations:                    174   AIC:                            -79.61
Df Residuals:                        156   BIC:                            -22.75
Df Model:                             17                                         
Covariance Type:               nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------

In [6]:
# Chapter 9: descriptive in-sample error on the regression sample
y_hat = ols_model.fittedvalues
mae = np.mean(np.abs(y - y_hat))
rmse = np.sqrt(np.mean((y - y_hat) ** 2))
print(f"In-sample MAE: {mae:.4f} | In-sample RMSE: {rmse:.4f}")
print(f"R-squared: {ols_model.rsquared:.4f} | Adj. R-squared: {ols_model.rsquared_adj:.4f}")


In-sample MAE: 0.1424 | In-sample RMSE: 0.1736
R-squared: 0.3553 | Adj. R-squared: 0.2851


In [7]:
# Chapter 10: VIF for case-mix matrix only (safehouse FE dummies omitted — many categories)
X_case = pd.concat([X_num, X_cat], axis=1).astype(float)
vif_rows = []
for i in range(X_case.shape[1]):
    vif_rows.append(
        {"feature": X_case.columns[i], "VIF": variance_inflation_factor(X_case.values, i)}
    )
vif_df = pd.DataFrame(vif_rows).sort_values("VIF", ascending=False)
print("VIF (case-mix predictors, no intercept):")
print(vif_df.to_string())

# Residual diagnostics
resid = ols_model.resid
fitted = ols_model.fittedvalues

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(fitted, resid, alpha=0.7)
axes[0].axhline(0, color="black", lw=1)
axes[0].set_title("Residuals vs fitted")
axes[0].set_xlabel("Fitted values")
axes[0].set_ylabel("Residuals")

sm.qqplot(resid, line="45", ax=axes[1])
axes[1].set_title("Normal Q-Q (residuals)")
plt.tight_layout()
plt.show()


VIF (case-mix predictors, no intercept):
                     feature       VIF
2   age_upon_admission_years  8.971863
5  initial_risk_level_Medium  4.787372
3    initial_risk_level_High  3.938316
4     initial_risk_level_Low  2.931847
8  case_category_Surrendered  2.467180
7    case_category_Neglected  1.778605
1              family_is_4ps  1.688782
6    case_category_Foundling  1.685071
0                     is_pwd  1.125855


In [8]:
# Safehouse value-add table (coefficients + p-values)
param_names = [p for p in ols_model.params.index if str(p).startswith("safehouse_id_")]
safehouse_effects = (
    pd.DataFrame(
        {
            "term": param_names,
            "coef": ols_model.params[param_names].values,
            "pvalue": ols_model.pvalues[param_names].values,
        }
    )
    .assign(
        safehouse_id=lambda d: d["term"].str.replace("safehouse_id_", "", regex=False).astype(int)
    )
    .merge(safehouses[["safehouse_id", "name", "safehouse_code"]], on="safehouse_id", how="left")
    .sort_values("coef")
)

# Reference safehouse (dropped dummy): coefficient 0 by construction
ref_id = int(sorted(reg["safehouse_id"].dropna().unique())[0])
ref_row = pd.DataFrame(
    [
        {
            "term": f"(reference: safehouse_id={ref_id})",
            "coef": 0.0,
            "pvalue": np.nan,
            "safehouse_id": ref_id,
            "name": safehouses.loc[safehouses["safehouse_id"].eq(ref_id), "name"].iloc[0],
            "safehouse_code": safehouses.loc[safehouses["safehouse_id"].eq(ref_id), "safehouse_code"].iloc[0],
        }
    ]
)

safehouse_effects_all = pd.concat([ref_row, safehouse_effects], ignore_index=True)
safehouse_effects_all


,term,coef,pvalue,safehouse_id,name,safehouse_code
0,(reference: safehouse_id=1),0.000000,NaN,1,Lighthouse Safehouse 1,SH01
1,safehouse_id_7,-0.271157,0.000006,7,Lighthouse Safehouse 7,SH07
2,safehouse_id_9,-0.258859,0.005823,9,Lighthouse Safehouse 9,SH09
3,safehouse_id_5,-0.190509,0.002550,5,Lighthouse Safehouse 5,SH05
4,safehouse_id_4,-0.168156,0.009720,4,Lighthouse Safehouse 4,SH04
5,safehouse_id_2,-0.159136,0.025475,2,Lighthouse Safehouse 2,SH02
6,safehouse_id_6,-0.095783,0.126507,6,Lighthouse Safehouse 6,SH06
7,safehouse_id_3,0.052279,0.337950,3,Lighthouse Safehouse 3,SH03
8,safehouse_id_8,0.085676,0.145439,8,Lighthouse Safehouse 8,SH08


## 4. Evaluation & Interpretation

We relate **estimated safehouse value-add** (safehouse coefficients, including the reference category at **0**) to **total Operations budget** per safehouse, then quantify association with **Pearson** and **Spearman** correlation.

This section is **evaluative** for the business question (“does funding follow true value-add?”), not a claim that the regression’s predictive accuracy is the primary success metric (**Chapter 11** differs).


In [9]:
# Attach budget + coefs for plotting
coef_map = dict(zip(safehouse_effects_all["safehouse_id"], safehouse_effects_all["coef"]))
plot_df = ops_budget_by_sh.copy()
plot_df["value_add_coef"] = plot_df["safehouse_id"].map(coef_map)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(plot_df["value_add_coef"], plot_df["operations_budget_total"], s=80, alpha=0.85)
for _, r in plot_df.iterrows():
    ax.annotate(
        str(int(r["safehouse_id"])),
        (r["value_add_coef"], r["operations_budget_total"]),
        textcoords="offset points",
        xytext=(4, 4),
        fontsize=9,
    )
ax.set_xlabel("Safehouse value-add coefficient (vs reference safehouse)")
ax.set_ylabel("Total Operations budget allocated")
ax.set_title("Value-add vs Operations budget (safehouse level)")
plt.tight_layout()
plt.show()

mask = plot_df["value_add_coef"].notna() & plot_df["operations_budget_total"].notna()
pearson_r, pearson_p = stats.pearsonr(plot_df.loc[mask, "value_add_coef"], plot_df.loc[mask, "operations_budget_total"])
spearman_r, spearman_p = stats.spearmanr(plot_df.loc[mask, "value_add_coef"], plot_df.loc[mask, "operations_budget_total"])

print(f"Pearson r = {pearson_r:.3f} (p = {pearson_p:.4f})")
print(f"Spearman rho = {spearman_r:.3f} (p = {spearman_p:.4f})")


Pearson r = 0.596 (p = 0.0905)
Spearman rho = 0.617 (p = 0.0769)


### Interpretation (quadrants)

Use the scatter to locate **high vs low value-add** (horizontal axis) and **high vs low Operations funding** (vertical axis). Facilities in **high value-add / low funding** can signal efficiency (or measurement issues); **low value-add / high funding** can motivate audits or operational support. Correlation summarizes **linear / monotonic alignment** between the case-mix-adjusted ranking proxy and budget — it does **not** prove that funding *caused* outcomes (see Section 5).


## 5. Causal and Relationship Analysis

### Case-mix coefficients (why adjustment matters)
Inspect the OLS table for `initial_risk_level` and `case_category` dummy coefficients. With `drop_first=True`, the **omitted reference** levels are **`initial_risk_level == Critical`** and **`case_category == Abandoned`**. Coefficients on the remaining levels are **differences relative to those baselines**. If case-mix indicators shift the outcome in plausible directions, that supports the claim that **composition differs across safehouses** and naive facility rankings can be misleading.

### Chapter 10 diagnostics and cautious language
If **VIF** indicates multicollinearity among case-mix predictors, or **heteroscedasticity** appears in residuals vs fitted, standard errors and p-values should be interpreted cautiously. Diagnostics are **signals** guiding judgment, not automatic disqualification.

### Omitted-variable bias (OVB)
Even with measured case mix, unobserved factors may still confound the **safehouse effect**: e.g., local community resources, staffing stability/tenure, referral network quality, unmeasured trauma severity, and policy changes over time.

### Budget vs value-add correlation is not causal
A positive correlation between **Operations** totals and estimated value-add **does not** imply that increasing the budget *causes* better outcomes. It is at best an **allocative efficiency** diagnostic: whether funding tracks measured performance after case-mix adjustment.


In [10]:
# Export safehouse value-add scores to Supabase (service role key required)
import os

try:
    from supabase import create_client
except ImportError as exc:
    raise ImportError("Install supabase client first: pip install supabase") from exc

supabase_url = os.getenv("SUPABASE_URL")
supabase_service_key = os.getenv("SUPABASE_SERVICE_KEY") or os.getenv("SUPABASE_SERVICE_ROLE_KEY")

if not supabase_url or not supabase_service_key:
    raise EnvironmentError(
        "Missing SUPABASE_URL or SUPABASE_SERVICE_KEY/SUPABASE_SERVICE_ROLE_KEY in environment."
    )

# Aggregate raw average change and resident volume for display context
safehouse_rollup = (
    model_df.dropna(subset=["safehouse_id", "health_score_delta_6m"])
    .groupby("safehouse_id", as_index=False)
    .agg(
        avg_outcome_change=("health_score_delta_6m", "mean"),
        resident_count=("resident_id", pd.Series.nunique),
    )
)

# Use OLS safehouse coefficients already computed above
export_df = (
    safehouse_effects_all[["safehouse_id", "name", "coef"]]
    .rename(columns={"name": "safehouse_name", "coef": "value_add_coef"})
    .merge(safehouse_rollup, on="safehouse_id", how="left")
)

export_df["model_version"] = "1.0"

rows = export_df[
    [
        "safehouse_id",
        "safehouse_name",
        "value_add_coef",
        "avg_outcome_change",
        "resident_count",
        "model_version",
    ]
].to_dict(orient="records")

client = create_client(supabase_url, supabase_service_key)
resp = client.table("safehouse_ml_scores").upsert(rows, on_conflict="safehouse_id").execute()

print(f"Upserted {len(rows)} rows to public.safehouse_ml_scores")
if getattr(resp, "data", None) is not None:
    print(f"Response rows: {len(resp.data)}")

Upserted 9 rows to public.safehouse_ml_scores
Response rows: 9


## 6. Deployment Notes

### Resource allocation dashboard (Executive / Finance)
A practical deployment is a **web dashboard** that plots each safehouse on the same two axes used above: **case-mix-adjusted value-add** (from the fixed-effects regression) vs **total Operations allocation**. The board can be required to **justify** continued high funding for low value-add facilities, or to investigate **why** some facilities achieve strong outcomes with modest Operations inflows.

### Governance workflow
- **Quarterly refresh**: re-run this notebook when new monthly health records and allocations arrive.
- **Audit triggers**: automatic flags for **low value-add / high funding** quadrants; optional drill-down into staffing and incident metrics (outside this notebook’s scope).

This turns the analysis from a one-off chart into an ongoing **resource allocation** conversation tied to transparent modeling assumptions and limitations.


## Deployment Notes

**Portal route:** `/portal/reports`  
**Component:** `ReportsPage` in `src/pages/portal/ReportsPage.tsx`  
**Supabase table:** `public.safehouse_ml_scores`

### Supabase table schema

```sql
CREATE TABLE IF NOT EXISTS public.safehouse_ml_scores (
  id               bigint GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  safehouse_id     integer NOT NULL,
  safehouse_name   text,
  value_add_coef   numeric(8, 4),   -- OLS coefficient (case-mix-adjusted)
  avg_outcome_change numeric(8, 4), -- raw average delta for reference
  resident_count   integer,
  model_version    text NOT NULL DEFAULT '1.0',
  scored_at        timestamptz NOT NULL DEFAULT now(),
  UNIQUE (safehouse_id)
);
```

### How scores reach the UI

1. Extract the OLS safehouse indicator coefficients from the fitted model:

```python
# After fitting OLS with safehouse dummies, extract coefficients
safehouse_coefs = pd.DataFrame({
    "safehouse_id": [int(c.replace("safehouse_", "")) for c in coef_cols],
    "value_add_coef": results.params[coef_cols].values
})
# Merge in names and resident counts
safehouse_coefs = safehouse_coefs.merge(safehouse_meta, on="safehouse_id", how="left")
safehouse_coefs["safehouse_name"] = safehouse_coefs["name"]
safehouse_coefs["model_version"] = "1.0"
```

2. Export to Supabase:

```python
from supabase import create_client
import os

client = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_KEY"])

rows = safehouse_coefs[["safehouse_id", "safehouse_name", "value_add_coef",
                         "resident_count", "model_version"]].to_dict(orient="records")
client.table("safehouse_ml_scores").upsert(rows, on_conflict="safehouse_id").execute()
print(f"Upserted {len(rows)} rows.")
```

### Where it appears in the app

- **Reports page → Safehouse Value-Add Analysis card:** horizontal bar chart showing each safehouse's case-mix-adjusted coefficient. Staff and admin only.

### Explanatory framing note

This notebook is **explanatory**, not predictive (Ch. 1 distinction). We are not predicting future outcomes — we are estimating conditional associations using OLS with case-mix controls. Coefficients should be interpreted directionally, not causally: omitted variables and selection effects (who gets admitted where) cannot be ruled out with observational data. This is stated explicitly in the portal UI disclaimer.